# GUV Phase Partitioning Analysis Pipeline

## Publication Reference
This computational pipeline is the official implementation for the analysis presented in:
> **Hierarchy of Hydrophobic and Electrostatic Interactions in DNA–Membrane Phase Selectivity**  
> *Siu Ho Wong, Yameng Lou, Yuduo Chen, Diana Morzy, and Maartje M.C. Bastings*  
> **ACS Applied Materials & Interfaces 2025** 17 (46), 63871–63881  
> [DOI: 10.1021/acsami.5c13271](https://doi.org/10.1021/acsami.5c13271)

## Overview
Quantifies spatial partitioning of fluorescent markers in giant unilamellar vesicles (GUVs) exhibiting liquid-disordered ($L_d$) and liquid-ordered ($L_o$) phase separation using dual-channel confocal microscopy. Channel 0 = Liss Rhod (membrane); Channel 1 = Cy5-DNA (marker).

## Pipeline Stages
1. **Detection:** Gaussian smoothing → Canny edges → Hough Circle Transform.
2. **Masking:** Ring mask (enlarged outer, inner cutoff) refined by DNA-channel least-squares circle fit.
3. **Segmentation:** Intensity thresholding + Angular Gap Analysis → $L_d$ / $L_o$ arc delineation.
4. **Post-processing:** Morphological closing → skeletonisation → dilation for contour sampling.
5. **Quantification:** Mean/max intensities, area metrics, and geometric metadata per vesicle.
6. **Visualisation:** 4-panel QC figure saved per image (TIFF, 300 dpi).

In [ ]:
import os
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from skimage.io import imread
from skimage.filters import gaussian, threshold_otsu
from skimage.morphology import (
    binary_dilation, disk, closing, remove_small_objects, skeletonize
)
from scipy.ndimage import label
from scipy.optimize import least_squares

In [ ]:
# =============================================================================
# SECTION 1 — PATHS & OUTPUT CONFIGURATION
# =============================================================================
# Set base_dir to the root of your data folder, or use os.getcwd() when the
# notebook sits alongside the data/ directory.
base_dir     = os.getcwd()
image_folder = os.path.join(base_dir, 'data', 'raw')
output_dir   = os.path.join(base_dir, 'data', 'processed')
file_output  = os.path.join(output_dir, 'Analysis_Results.csv')

os.makedirs(output_dir, exist_ok=True)
print(f'Input  folder : {image_folder}')
print(f'Output folder : {output_dir}')

# =============================================================================
# SECTION 2 — ADJUSTABLE PARAMETERS
# =============================================================================

# --- Ring mask geometry ---
enlargement_factor = 1.2   # Outer radius = detected_radius x factor
inner_percentage   = 0.7   # Inner radius = outer_radius x percentage

# --- Ld/Lo threshold method ---
# Options: 'otsu' | 'mean' | 'median' | 'percentile' | 'fixed' | 'mean_plus_std'
threshold_method = 'otsu'
fixed_threshold  = 100     # Only used when threshold_method = 'fixed'

# --- Post-processing ---
min_arc_size   = 50        # Minimum pixel count to retain a bright cluster
closing_radius = 3         # Morphological closing disk radius for Ld/Lo masks
skeleton_width = 1         # Dilation disk radius on skeletonised contours

# --- DNA mask refinement ---
dna_dilation_radius = 0    # disk(0) = identity; increase to expand DNA mask

# --- Hough Circle Transform ---
hough_min_dist   = 80
hough_param1     = 50
hough_param2     = 30
hough_min_radius = 10
hough_max_radius = 100

In [ ]:
# =============================================================================
# SECTION 3 — HELPER FUNCTIONS
# =============================================================================

def compute_threshold(pixels, method, fixed_value=None):
    """
    Compute an intensity threshold for Ld/Lo segmentation.

    Parameters
    ----------
    pixels      : 1-D array of pixel intensities within the ring mask.
    method      : str — 'otsu' | 'mean' | 'median' | 'percentile' |
                  'fixed' | 'mean_plus_std'.
    fixed_value : float — required when method='fixed'.

    Returns
    -------
    float threshold value.
    """
    if len(pixels) == 0:
        return 0
    if method == 'otsu':
        return threshold_otsu(pixels)
    elif method == 'mean':
        return np.mean(pixels)
    elif method == 'median':
        return np.median(pixels)
    elif method == 'percentile':
        return np.percentile(pixels, 75)
    elif method == 'fixed':
        if fixed_value is None:
            raise ValueError("'fixed_threshold' must be set when method='fixed'.")
        return fixed_value
    elif method == 'mean_plus_std':
        return np.mean(pixels) + np.std(pixels)
    else:
        raise ValueError(f"Unknown threshold method: '{method}'.")


def fit_circle(points):
    """
    Fit a circle to a set of 2-D points using least-squares optimisation.

    Parameters
    ----------
    points : (N, 2) array of (x, y) coordinates.

    Returns
    -------
    xc, yc : float — fitted circle centre.
    R      : float — fitted circle radius.
    """
    def calc_R(xc, yc):
        return np.sqrt((points[:, 0] - xc) ** 2 + (points[:, 1] - yc) ** 2)

    def residuals(c):
        Ri = calc_R(*c)
        return Ri - Ri.mean()

    center_estimate = points.mean(axis=0)
    result = least_squares(residuals, center_estimate)
    xc, yc = result.x
    R = calc_R(xc, yc).mean()
    return xc, yc, R


def get_ld_arc_mask(ring_yy, ring_xx, valid_pixels, cx, cy, image_shape):
    """
    Identify the dominant Ld arc via Angular Gap Analysis.

    Sorts angular positions of bright (Ld-candidate) pixels around the vesicle
    centre, locates the largest angular discontinuity (= Lo gap), and returns
    the complementary arc as the Ld mask. Correctly handles arcs that cross
    the 0 / 2pi wrap boundary.

    Parameters
    ----------
    ring_yy, ring_xx : coordinate arrays from np.where(ring_mask).
    valid_pixels     : bool array — True for Ld-candidate pixels.
    cx, cy           : float — vesicle centre coordinates.
    image_shape      : tuple — (H, W) of the source image.

    Returns
    -------
    Ld_arc_mask : bool array of shape image_shape.
    """
    angles = np.arctan2(ring_yy - cy, ring_xx - cx)
    angles = (angles + 2 * np.pi) % (2 * np.pi)   # map to [0, 2pi)
    valid_angles = angles[valid_pixels]

    Ld_arc_mask = np.zeros(image_shape, dtype=bool)
    if len(valid_angles) == 0:
        return Ld_arc_mask

    sorted_angles = np.sort(valid_angles)
    diffs     = np.diff(sorted_angles)
    wrap_diff = (sorted_angles[0] - sorted_angles[-1] + 2 * np.pi) % (2 * np.pi)
    max_gap   = diffs.max() if len(diffs) > 0 else 0

    if wrap_diff > max_gap:
        # Largest gap wraps across the 0/2pi boundary
        start_angle = sorted_angles[0]
        end_angle   = sorted_angles[-1]
    else:
        max_gap_idx = np.argmax(diffs)
        start_angle = sorted_angles[max_gap_idx + 1]
        end_angle   = sorted_angles[max_gap_idx]

    # Normalise to [0, 2pi) BEFORE comparison (fixes PRO version bug)
    start_angle %= 2 * np.pi
    end_angle   %= 2 * np.pi

    if start_angle <= end_angle:
        in_ld = (angles >= start_angle) & (angles <= end_angle)
    else:
        # Arc crosses the 0/2pi wrap boundary
        in_ld = (angles >= start_angle) | (angles <= end_angle)

    Ld_arc_mask[ring_yy[in_ld], ring_xx[in_ld]] = True
    return Ld_arc_mask

In [ ]:
# =============================================================================
# SECTION 4 — MAIN PROCESSING LOOP
# =============================================================================
if not os.path.exists(image_folder):
    raise FileNotFoundError(
        f'Input folder not found: {image_folder}\n'
        'Update image_folder in Section 1 to point to your data.'
    )

image_counter = 0
print(f'Starting analysis on: {image_folder}\n')

for library in sorted(os.listdir(image_folder)):
    lib_path = os.path.join(image_folder, library)
    if not (os.path.isdir(lib_path) and '-d84' in library):
        continue

    for image_filename in sorted(os.listdir(lib_path)):
        if 'ome.tif' not in image_filename:
            continue

        image_counter += 1
        print(f'[{image_counter}] Processing: {library}/{image_filename}')

        # ------------------------------------------------------------------
        # 4.1  Load channels
        # ------------------------------------------------------------------
        image_multi = imread(os.path.join(lib_path, image_filename))
        image     = image_multi[0].copy()   # Ch.0 — Liss Rhod (membrane / Ld)
        DNA_image = image_multi[1].copy()   # Ch.1 — Cy5 (DNA marker)

        # ------------------------------------------------------------------
        # 4.2  Vesicle detection: Gaussian -> Canny -> Hough
        # ------------------------------------------------------------------
        blurred = gaussian(image, sigma=1.5, preserve_range=True)
        edges   = cv2.Canny(np.uint8(blurred), 40, 120)
        circles = cv2.HoughCircles(
            edges, cv2.HOUGH_GRADIENT, dp=1,
            minDist=hough_min_dist,
            param1=hough_param1, param2=hough_param2,
            minRadius=hough_min_radius, maxRadius=hough_max_radius
        )
        if circles is None:
            print(f'   No circles detected — skipping.')
            continue
        circles = circles[0]   # shape: (N, 3) -> [x, y, r]

        # ------------------------------------------------------------------
        # 4.3  DNA channel mask for ring refinement (mask4)
        # ------------------------------------------------------------------
        blurred_dna = gaussian(DNA_image, sigma=2, preserve_range=True)
        mask4 = blurred_dna > threshold_otsu(blurred_dna)
        if dna_dilation_radius > 0:
            mask4 = binary_dilation(mask4, disk(dna_dilation_radius))

        # ------------------------------------------------------------------
        # 4.4  Initialise per-image data structures
        # ------------------------------------------------------------------
        labeled_mask = np.zeros_like(image, dtype=np.uint16)   # vesicle ID map
        all_mask_Ld  = np.zeros_like(image, dtype=bool)
        all_mask_Lo  = np.zeros_like(image, dtype=bool)
        df_final     = pd.DataFrame()

        # ------------------------------------------------------------------
        # 4.5  Per-vesicle processing
        # ------------------------------------------------------------------
        for i, (x, y, r) in enumerate(circles, start=1):

            # -- Ring mask ------------------------------------------------
            enlarged_radius = r * enlargement_factor
            inner_radius    = enlarged_radius * inner_percentage
            yy, xx  = np.ogrid[:image.shape[0], :image.shape[1]]
            dist_sq = (yy - y) ** 2 + (xx - x) ** 2
            ring_mask = (dist_sq <= enlarged_radius ** 2) & (dist_sq > inner_radius ** 2)

            # -- DNA-guided circle fitting --------------------------------
            valid_pixels_mask = mask4 & ring_mask
            valid_points = np.column_stack(np.where(valid_pixels_mask))[:, [1, 0]]

            if len(valid_points) > 3:
                xc_fit, yc_fit, r_fit = fit_circle(valid_points)
                fit_dist_sq = (yy - yc_fit) ** 2 + (xx - xc_fit) ** 2
                final_ring_mask = (
                    (fit_dist_sq <= r_fit ** 2) &
                    (fit_dist_sq > (r_fit - 1) ** 2)
                )
            else:
                final_ring_mask = ring_mask

            if np.any(final_ring_mask):
                labeled_mask[final_ring_mask] = i

            # -- Ld/Lo intensity thresholding ----------------------------
            ring_yy, ring_xx = np.where(ring_mask)
            pixels_in_ring   = image[ring_mask]

            if len(pixels_in_ring) > 0:
                thresh       = compute_threshold(pixels_in_ring, threshold_method, fixed_threshold)
                bright_mask_full = np.zeros_like(image, dtype=bool)
                bright_mask_full[ring_yy, ring_xx] = pixels_in_ring > thresh

                labeled_bright, num_labels = label(
                    bright_mask_full, structure=np.ones((3, 3), dtype=int)
                )
                if num_labels > 0:
                    filtered = remove_small_objects(labeled_bright, min_size=min_arc_size)
                    valid_pixels = filtered[ring_yy, ring_xx] > 0
                else:
                    valid_pixels = np.zeros(len(ring_yy), dtype=bool)
            else:
                valid_pixels = np.zeros(len(ring_yy), dtype=bool)

            # -- Angular Gap Analysis -> Ld arc --------------------------
            Ld_arc_mask = get_ld_arc_mask(
                ring_yy, ring_xx, valid_pixels, x, y, image.shape
            )

            # -- Ld / Lo region masks ------------------------------------
            mask_Ld = closing(final_ring_mask & Ld_arc_mask,  disk(closing_radius))
            mask_Lo = closing(final_ring_mask & ~Ld_arc_mask, disk(closing_radius))

            # -- Skeletonisation + dilation for contour sampling ---------
            skel_Ld = binary_dilation(skeletonize(mask_Ld), disk(skeleton_width))
            skel_Lo = binary_dilation(skeletonize(mask_Lo), disk(skeleton_width))
            all_mask_Ld |= skel_Ld
            all_mask_Lo |= skel_Lo

            # -- Measurements --------------------------------------------
            area_total = int(np.sum(final_ring_mask))
            area_Ld    = int(np.sum(mask_Ld))
            area_Lo    = int(np.sum(mask_Lo))

            int_Ld  = DNA_image[skel_Ld]
            int_Lo  = DNA_image[skel_Lo]
            int_all = DNA_image[skel_Ld | skel_Lo]

            mean_Ld  = float(np.mean(int_Ld))  if area_Ld > 0 else float('nan')
            max_Ld   = float(np.max(int_Ld))   if area_Ld > 0 else float('nan')
            mean_Lo  = float(np.mean(int_Lo))  if area_Lo > 0 else float('nan')
            max_Lo   = float(np.max(int_Lo))   if area_Lo > 0 else float('nan')
            mean_all = float(np.mean(int_all)) if (area_Ld + area_Lo) > 0 else float('nan')

            # -- Append to DataFrame -------------------------------------
            df_final = pd.concat([df_final, pd.DataFrame({
                'vesicle_label'   : i,
                'library'         : library,
                'center_x'        : int(x),
                'center_y'        : int(y),
                'radius'          : int(r),
                'enlarged_radius' : enlarged_radius,
                'inner_radius'    : inner_radius,
                'area_total'      : area_total,
                'area_Ld'         : area_Ld,
                'area_Lo'         : area_Lo,
                'mean_DNA(Ld)'    : mean_Ld,
                'max_DNA(Ld)'     : max_Ld,
                'mean_DNA(Lo)'    : mean_Lo,
                'max_DNA(Lo)'     : max_Lo,
                'mean_DNA'        : mean_all,
            }, index=[0])], ignore_index=True)

        # ------------------------------------------------------------------
        # 4.6  Save CSV (write header on first image, append thereafter)
        # ------------------------------------------------------------------
        df_final.to_csv(
            file_output,
            mode='w' if image_counter == 1 else 'a',
            index=False,
            header=(image_counter == 1)
        )

        # ------------------------------------------------------------------
        # 4.7  Visualisation — 4-panel QC figure
        # ------------------------------------------------------------------
        fig, axes = plt.subplots(1, 4, figsize=(16, 5))
        fig.suptitle(f'{library} / {image_filename}', fontsize=9)

        # Panel 0: raw image + Hough circle overlays
        axes[0].imshow(image, cmap='gray')
        for (cx, cy, cr) in circles:
            axes[0].add_patch(
                plt.Circle((cx, cy), cr * enlargement_factor,
                            color='red', fill=False, linewidth=0.8)
            )
        axes[0].set_title('Detected GUVs')

        # Panel 1: labelled ring mask
        axes[1].imshow(labeled_mask, cmap='gray')
        axes[1].set_title('Final Ring Mask')

        # Panels 2 & 3: Ld/Lo overlays on both channels
        valid_labels = np.unique(labeled_mask)
        valid_labels = valid_labels[valid_labels > 0]

        for ax_idx, (base_img, ch_lbl) in enumerate(
            [(image, 'Ch.1 GUV'), (DNA_image, 'Ch.2 DNA')], start=2
        ):
            axes[ax_idx].imshow(base_img, cmap='gray')
            axes[ax_idx].imshow(all_mask_Ld, cmap='Greens', alpha=0.5)
            axes[ax_idx].imshow(all_mask_Lo, cmap='Reds',   alpha=0.5)
            axes[ax_idx].set_title(f'Ld (green) / Lo (red) — {ch_lbl}')
            for lbl in valid_labels:
                cx, cy, cr = circles[lbl - 1]
                axes[ax_idx].text(
                    int(cx + enlarged_radius + 10), int(cy), str(lbl),
                    color='white', fontsize=7, ha='left', va='center',
                    bbox=dict(boxstyle='round,pad=0.1', fc='black', alpha=0.4)
                )

        for ax in axes:
            ax.axis('off')
        plt.tight_layout()

        fig_name = f'{library}_{image_filename[:-4]}_analysis.tif'
        fig.savefig(os.path.join(output_dir, fig_name), dpi=300, bbox_inches='tight')
        plt.show()
        plt.close()

print(f'\n--- Analysis complete. {image_counter} image(s) processed. ---')
print(f'Results saved to: {file_output}')

## Quick Result Preview
Run the cell below after the main loop to inspect the aggregated output table.

In [ ]:
# =============================================================================
# SECTION 5 — RESULT PREVIEW
# =============================================================================
if os.path.exists(file_output):
    df_all = pd.read_csv(file_output)
    print(f'Total vesicles analysed : {len(df_all)}')
    print(f'Columns                 : {list(df_all.columns)}')
    display(df_all.head(10))
    summary = df_all.groupby('library')[['mean_DNA(Ld)', 'mean_DNA(Lo)', 'mean_DNA']].describe()
    display(summary)
else:
    print('No results file found — run Section 4 first.')